# Session 20 lab: a neuron, a training loop and a network on the loans

**AI and Data Science Professional** | KLAi Professional Learning

You build one artificial neuron and a gradient descent loop with numpy, spend ten minutes in TensorFlow Playground, then compare an MLPClassifier with a random forest and logistic regression on the Nile Microfinance loans. You finish with a results table and a three-sentence recommendation for the credit team.

**How to use this notebook**

1. Run the cells in order from the top (Shift and Enter).
2. Every step has the same rhythm: **PREDICT**, run, **READ**, then **CHANGE** one thing.
3. Cells that say `answer = None` are yours to fill in. Replace `None` with your code. Then run the check cell under it.
4. The neuron and the weights in Step 1 are illustrative (made up to show the arithmetic). The loans are fictional and generated.
5. The AUC checks accept a range, because scikit-learn starts from random weights. If a check says NOT YET and your number is close, read the range in the check cell.
6. If you get an error, read the last line first.

In [ ]:
# Run this cell first. It puts the course data files next to your notebook.
import os, re, zipfile, urllib.request
DATA_LINK = "https://drive.google.com/file/d/11Z5tr2iKuPROn5cyi9P7me6nd3naKfNp/view?usp=sharing"   # the Google Drive link to data.zip that your tutor shares (leave empty to upload data.zip yourself)
if not os.path.exists("orders.csv"):
    if DATA_LINK:
        m = re.search(r"/d/([A-Za-z0-9_-]+)|[?&]id=([A-Za-z0-9_-]+)", DATA_LINK)
        file_id = (m.group(1) or m.group(2)) if m else DATA_LINK
        urllib.request.urlretrieve("https://drive.google.com/uc?export=download&id=" + file_id, "data.zip")
    else:
        from google.colab import files
        print("Choose the file data.zip from your computer")
        uploaded = files.upload()
        os.replace(list(uploaded)[0], "data.zip")   # works even if the browser renamed it to data (1).zip
    zipfile.ZipFile("data.zip").extractall(".")
print("Data files ready:", sorted(f for f in os.listdir(".") if f.endswith((".csv", ".db"))))

In [ ]:
# Automatic check helper. Run this cell once. You do not need to read it.
_results = {}   # label -> True or False; running a check again replaces its old result
def check(label, value, expected, tol=0.01):
    try:
        import numpy as _np
        ok = bool(_np.all(_np.abs(_np.asarray(value, dtype=float) - _np.asarray(expected, dtype=float)) <= tol))
    except Exception:
        ok = False
    _results[label] = ok
    print(("OK       " if ok else "NOT YET  ") + label)
def show_checks():
    n = len(_results); k = sum(_results.values())
    print("All %d checks pass" % n if k == n and n else "%d of %d checks pass" % (k, n))

In [ ]:
# A helper so that a check never crashes while your answer is still None. You do not need to read it.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def safe(f):
    try:
        return float(f())
    except Exception:
        return float("nan")

---
## Step 1: One artificial neuron

A neuron is a weighted sum of the inputs plus a bias, passed through an activation function. In a spreadsheet the weighted sum is SUMPRODUCT.

In [ ]:
# one loan: late payments, savings (UGX millions), loan size (UGX millions)
x = np.array([1, 0.8, 0.3])
w = np.array([0.9, -1.2, 1.5])   # illustrative weights that were never trained
b = -1.0                         # the bias
print("inputs", x, "weights", w, "bias", b)

**PREDICT first.** Will the sigmoid score of this loan be above 0.5 or below 0.5? Work out the weighted sum plus bias by hand first.

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
answer = None  # your code here: np.dot(x, w) + b is the weighted sum plus bias. Put it into the sigmoid 1 / (1 + np.exp(-z))

In [ ]:
try:
    score = answer
    print("sigmoid score:", round(float(score), 3))
except Exception as err:
    print("Not yet: an answer above is still None or incomplete (" + type(err).__name__ + ").")

In [ ]:
check("Step 1a: neuron score", safe(lambda: answer), 0.3521, 0.001)

**READ the output.** Which input pushes the score up and which pushes it down? Where does the weighted sum plus bias come out, and what does that mean for the sigmoid?

> My answer: 

In [ ]:
answer = None  # your code here: ReLU returns 0 for a negative total and the total itself otherwise

In [ ]:
try:
    relu_out = answer
    print("ReLU output:", relu_out)
except Exception as err:
    print("Not yet: an answer above is still None or incomplete (" + type(err).__name__ + ").")

In [ ]:
check("Step 1b: ReLU of a negative total is 0", safe(lambda: answer), 0.0, 0.001)

**CHANGE one thing.** Make the first weight larger: use w2 = np.array([4.5, -1.2, 1.5]) in the cell below. What does the score become? What does that say about late payments?

> What I saw: 

In [ ]:
answer = None  # your code here: the same neuron with the first weight 4.5

In [ ]:
try:
    new_score = answer
    print("new score:", round(float(new_score), 3))
except Exception as err:
    print("Not yet: an answer above is still None or incomplete (" + type(err).__name__ + ").")

In [ ]:
check("Step 1c: the score is now above 0.9", safe(lambda: answer > 0.9), 1)

---
## Step 2: A training loop on one weight

One weight must learn that 2 times the weight gives 6. The loss is (2w - 6) squared. Gradient descent moves the weight against the slope of the loss.

**PREDICT first.** Start at w = 0 with a learning rate of 0.05 and run 8 steps. Will w end above 3, below 3, or exactly 3?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
answer = None  # your code here: a loop of 8 steps: find the gradient, then w = w - lr * grad

In [ ]:
try:
    w_after = answer
    print("w after 8 steps:", round(float(w_after), 3))
except Exception as err:
    print("Not yet: an answer above is still None or incomplete (" + type(err).__name__ + ").")

In [ ]:
check("Step 2a: w after 8 steps at learning rate 0.05", safe(lambda: answer), 2.9496, 0.01)

**READ the output.** Is w exactly 3? Why do the steps get smaller as w approaches 3?

> My answer: 

**CHANGE one thing.** Run the same loop with lr = 0.25 and print w at every step. What do you see? Then try lr = 0.5.

> What I saw: 

In [ ]:
answer = None  # your code here: the same loop with lr = 0.5, then the loss (2w - 6) squared at the end

In [ ]:
try:
    loss_big = answer
    print("loss after the 8th update at lr = 0.5:", round(float(loss_big)))
except Exception as err:
    print("Not yet: an answer above is still None or incomplete (" + type(err).__name__ + ").")

In [ ]:
check("Step 2b: the loss blows up (above 100)", safe(lambda: answer > 100), 1)

---
## Step 3: TensorFlow Playground (10 minutes)

Open TensorFlow Playground in a new browser tab (search for it: it is free and needs no account). Work through the four challenges on the slide. Record what you see in your own words.

**Challenge 1, circle.** No hidden layer, then one hidden layer with 4 neurons. What changed?

> What I saw: 

**Challenge 2, spiral.** Add layers and neurons. Which size was enough?

> What I saw: 

**Challenge 3, overfitting.** Raise the noise, use few training points, train for a long time. When did the training loss and the test loss separate?

> What I saw: 

**Challenge 4, learning rate and activation.** Which settings failed, which were slow, which were fast?

> What I saw: 

---
## Step 4: Prepare the loans

The Nile Microfinance loans: each record is one loan and default is 1 when the loan went bad. The preparation is the one from Sessions 14 to 17: the two impossible ages (0 and 199) become the median age, so all 8,000 loans stay, and `collection_calls` is left out because it is recorded only after a loan goes bad. One difference: a blank income stays blank here, because the imputer inside the Pipeline learns the median from the training records only (leak-free). Sessions 14 to 17 computed it on all loans before the split, on purpose, to keep things simple.

**PREDICT first.** How many loans will remain, and how many records will train and test with a 75 and 25 split? How many defaults will the stratified test set hold?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

loans = pd.read_csv("loans.csv")
print(loans.shape)

In [ ]:
answer = None  # your code here: replace the two impossible ages (under 18 or over 100) by the median of the valid ages (36). Keep every row

In [ ]:
try:
    loans = answer
    loans["income_missing"] = loans["monthly_income_ugx"].isna().astype(int)
    # instalment_to_income: loan x 1.15 / term / monthly income. The 1.15 (loan plus 15 percent interest and fees) is an ILLUSTRATIVE assumption, not in the data. A blank income gives a blank ratio.
    loans["instalment_to_income"] = loans["loan_amount_ugx"] * 1.15 / loans["term_months"] / loans["monthly_income_ugx"]
    cols = ["age", "monthly_income_ugx", "income_missing", "loan_amount_ugx", "term_months", "previous_loans", "previous_late_payments", "group_loan",
            "savings_balance_ugx", "has_mobile_money", "distance_to_branch_km", "instalment_to_income"]
    X = loans[cols]          # collection_calls is left out on purpose: it leaks the answer
    y = loans["default"]
except Exception as err:
    print("Not yet: an answer above is still None or incomplete (" + type(err).__name__ + ").")

In [ ]:
answer = None  # your code here: train_test_split of X and y with test_size=0.25, stratify=y and random_state=42 (assign the four parts; answer is X_train)

In [ ]:
X_train = answer

In [ ]:
try:
    print(len(X), len(X_train), len(X_test), "defaults in the test records:", y_test.sum())
    print(X_test.index[:10].tolist())   # the same test rows as Sessions 14 to 17
except Exception as err:
    print("Not yet: an answer above is still None or incomplete (" + type(err).__name__ + ").")

In [ ]:
check("Step 4a: loans kept", safe(lambda: len(X)), 8000)
check("Step 4b: training records", safe(lambda: len(X_train)), 6000)
check("Step 4c: defaults in the test records", safe(lambda: y_test.sum()), 251)
check("Step 4d: same test rows as the course (sum of the row numbers)", safe(lambda: sum(X_test.index)), 8081998)
check("Step 4e: collection_calls is not in X", safe(lambda: "collection_calls" not in X.columns), 1)

**READ the output.** How many blank monthly_income_ugx values are there in X_train? Why must the network never see a blank, and why does the imputer sit inside the Pipeline?

> My answer: 

In [ ]:
answer = None  # your code here: weights plus biases of a network with 12 inputs, hidden layers of 16 and 8 neurons and 1 output

In [ ]:
try:
    n_params = answer
    print("parameters:", n_params)
except Exception as err:
    print("Not yet: an answer above is still None or incomplete (" + type(err).__name__ + ").")

In [ ]:
check("Step 4f: parameters of the 12-16-8-1 network", safe(lambda: answer), 353)

In [ ]:
answer = None  # your code here: the number of batches of 200 records in one pass over X_train

In [ ]:
try:
    n_batches = answer
    print("batches per epoch:", n_batches)
except Exception as err:
    print("Not yet: an answer above is still None or incomplete (" + type(err).__name__ + ").")

In [ ]:
check("Step 4g: batches per epoch", safe(lambda: answer), 30)

---
## Step 5: Logistic regression, a network and a forest on the same split

Every model sees the same training records and is scored on the same test records with the AUC. The imputer and the scaler sit inside the Pipeline, so they learn only from the training records. The logistic regression uses the default C = 1.0 (L2 penalty).

**PREDICT first.** Which will score the highest AUC on the test records: logistic regression, the MLPClassifier or the random forest? Write the order you expect.

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
answer = None  # your code here: make_pipeline with SimpleImputer(strategy="median"), StandardScaler() and LogisticRegression(max_iter=1000)

In [ ]:
try:
    logit = answer
    logit.fit(X_train, y_train)
    auc_logit = roc_auc_score(y_test, logit.predict_proba(X_test)[:, 1])
    print("logistic regression AUC:", round(auc_logit, 3))
except Exception as err:
    print("Not yet: an answer above is still None or incomplete (" + type(err).__name__ + ").")

In [ ]:
check("Step 5a: logistic regression AUC (reference 0.758, a range of 0.03)", safe(lambda: auc_logit), 0.7579, 0.03)

In [ ]:
answer = None  # your code here: make_pipeline with SimpleImputer, StandardScaler and MLPClassifier(hidden_layer_sizes=(16, 8), max_iter=500, random_state=42)

In [ ]:
try:
    mlp = answer
    mlp.fit(X_train, y_train)
    auc_mlp = roc_auc_score(y_test, mlp.predict_proba(X_test)[:, 1])
    print("MLPClassifier AUC:", round(auc_mlp, 3))
except Exception as err:
    print("Not yet: an answer above is still None or incomplete (" + type(err).__name__ + ").")

In [ ]:
check("Step 5b: MLPClassifier AUC (reference 0.742, a range of 0.05)", safe(lambda: auc_mlp), 0.7421, 0.05)

In [ ]:
answer = None  # your code here: make_pipeline with SimpleImputer and RandomForestClassifier(n_estimators=200, random_state=42). A forest needs no scaler

In [ ]:
try:
    forest = answer
    forest.fit(X_train, y_train)
    auc_rf = roc_auc_score(y_test, forest.predict_proba(X_test)[:, 1])
    print("random forest AUC:", round(auc_rf, 3))
except Exception as err:
    print("Not yet: an answer above is still None or incomplete (" + type(err).__name__ + ").")

In [ ]:
check("Step 5c: random forest AUC (between 0.69 and 0.77)", safe(lambda: auc_rf), 0.73, 0.04)

**READ the output.** Does the network beat logistic regression by enough to matter? Does the forest beat both? Is the order the one you predicted? Differences of up to about 0.03 from the slides are expected, because a scikit-learn network starts from random weights and trains with the Adam solver.

> My answer: 

**CHANGE one thing.** Add "collection_calls" to the list of variables, refit logistic regression and look at the AUC. Then remove it again and rerun the cells. Why must this variable stay out?

> What I saw: 

---
## Step 6: See overfitting in a network

Train a larger network on only 300 loans and compare its training AUC with its test AUC.

**PREDICT first.** A network with 64 and 64 hidden neurons has 300 training loans. Will its training AUC be near 0.7, or near 1.0? And its test AUC?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
answer = None  # your code here: the same kind of Pipeline with MLPClassifier(hidden_layer_sizes=(64, 64), max_iter=1000, random_state=42)

In [ ]:
try:
    big = answer
    big.fit(X_train.iloc[:300], y_train.iloc[:300])
    auc_train = roc_auc_score(y_train.iloc[:300], big.predict_proba(X_train.iloc[:300])[:, 1])
    auc_test = roc_auc_score(y_test, big.predict_proba(X_test)[:, 1])
    gap = auc_train - auc_test
    print("training AUC", round(auc_train, 3), "test AUC", round(auc_test, 3), "gap", round(gap, 3))
except Exception as err:
    print("Not yet: an answer above is still None or incomplete (" + type(err).__name__ + ").")

In [ ]:
check("Step 6: the training AUC is far above the test AUC (the numpy reference gap is 0.37)", safe(lambda: gap > 0.1), 1)

**READ the output.** Which of the two numbers would you quote to a manager? Name two ways to reduce this gap.

> My answer: 

---
## Step 7: Your results table and recommendation

Collect the three models in one table and save it. This file and your three sentences are what you keep. The file is saved in your Colab folder (the Files panel on the left): download it before you close Colab.

In [ ]:
answer = None  # your code here: a DataFrame with the columns model, test AUC and parameters or trees for the three models

In [ ]:
try:
    results = answer
    results.to_csv("session20_model_comparison.csv", index=False)   # saved in your Colab folder
    print(results)
except Exception as err:
    print("Not yet: an answer above is still None or incomplete (" + type(err).__name__ + ").")

In [ ]:
check("Step 7: the results table has three models", safe(lambda: len(answer)), 3)

In [ ]:
show_checks()

## Your recommendation to the credit team

Double-click this cell and write three sentences with your own numbers.

1. The three test AUC scores, from the same split.
2. Which model you recommend, and why (the simplest model that is good enough is a good answer).
3. One limit: for example one split only, the loans are fictional, or the AUC does not say how many borrowers you would approve.

> My three sentences: 

## Optional: a pretrained image classifier

Only if you finish early. The first run downloads the model weights from the Hugging Face Hub. The download is free, but read the model page first: the size and the licence can change. Upload only photos you are allowed to use, never photos of people or customers. The code is commented out; remove the # signs to run it.

In [ ]:
# !pip install -q transformers
# from transformers import pipeline
# from google.colab import files
# name = list(files.upload())[0]
# classifier = pipeline("image-classification")
# print(classifier(name)[:3])